<a href="https://colab.research.google.com/github/climatom/BioMetConference/blob/main/notebooks/02_HumidHeatImpactForecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Humid heat for impact-based forecasting

## Introduction

Humid heat--the combination of sensible and latent heat in the atmosphere--is a key control on the human heat balance. Interestingly, epidemiological models of heat-related mortality are not obviously improved by including a measure of atmospheric humidity ([Baldwin et al., 2023](https://pmc.ncbi.nlm.nih.gov/articles/PMC10231239/)). However, such statistical models are themselves unreliable for predicting health risks beyond the range of the observational data on which they are trained. In particular, the risk of death during unprecedented high temperatures could diverge substantially from statistical extrapolations because of tipping points in human thermal physiology ([Matthews et al., 2025](https://www.nature.com/articles/s43017-024-00635-w)).

The evidence is much more equivocal regarding the upper limits of human heat tolerance: the maximum level of humid heat that a person can tolerate before experiencing runaway overheating. These limits depend on the combination of air temperature and humidity. However, the jury is still very much out on *how* hot and *how* humid the air must become before lethal overheating occurs, even for a hypothetical population of young, healthy, acclimatised adults. The most widely used definition of *unsurvivable* heat ([Vanos et al., 2023](https://www.nature.com/articles/s41467-023-43121-5)) is almost certainly too conservative regarding the true limits of human heat tolerance ([Kearney et al., 2026](https://onlinelibrary.wiley.com/doi/10.1111/gcb.70830)). Nevertheless, proximity to such *unsurvivable* thresholds--even if their absolute values remain uncertain--may provide a useful measure of physiological strain that could, in principle, warn of potentially catastrophic heat mortality. I use the term *catastrophic* because humanity has almost certainly never crossed these unsurvivable limits ([Matthews et al., 2025](https://www.nature.com/articles/s43017-024-00635-w))--and yet heatwaves already kill many tens of thousands of people each year.

In this workbook, we will explore how to build an early warning system for *unsurvivable* heat. As well as providing a practical application, this will also give us the opportunity to compute several humid heat metrics, including some of those introduced in Workbook 1.

By the end of this session, you should be able to:


*   Programatically retreive weather forecast data using Herbie
*   Calculate humid heat metrics manually in Python
*   Use the heatindex Python module to compute wet-bulb temperature
*   Understand the concept of the 'enthalpy distance' to the unsurvivable threshold

Please now work through the notebook, running the code as you go.

## Download latest ECMWF forecast with Herbie

We begin by downloading the latest available ECMWF IFS deterministic forecast using [Herbie](https://herbie.readthedocs.io/).

Our aim here is simple: to retrieve the near-surface variables needed for humid-heat diagnostics.

We will download:

- 2-m air temperature (`2t`)
- 2-m dewpoint temperature (`2d`)
- surface pressure (`sp`)

The forecast is global and three-hourly. The forecast is available out to ten days (240 hours), but we will work with a smaller time horizon to keep data volumes manageable.


## Install and import libraries

In Google Colab, this cell installs the required packages. The next cell simply performs the necessary imports


In [ ]:
# Colab setup
# If running outside Colab, you may prefer to comment this out and install packages manually.
!pip -q install herbie-data cfgrib eccodes xarray netCDF4 tqdm heatindex cartopy

In [ ]:
# @title
from pathlib import Path
from datetime import datetime, timezone, timedelta
import time
import warnings
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.animation as animation
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm
from pathlib import Path
import heatindex as hi
from herbie import Herbie
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from IPython.display import HTML

warnings.filterwarnings("ignore")

## Settings

The main settings are collected here so that the notebook can be easily adapted.

At present, the notebook uses the deterministic ECMWF IFS open-data forecast. We also specify the temporal frequency of the forecast (`tres`) (which varies between forecasts products available through Herbie), and the maximum lead time that we want (`horz`). The options below will ask for a 3h forecast out to 72 hours. We set these in the following code block.

```python
model="ifs"
product="oper"
tres=3
horz=72
```

In [ ]:
# @title
# Output directory
# Configure access to data directory
OUTDIR = Path("/content/data/ecmwf_latest_forecast")
OUTDIR.mkdir(parents=True, exist_ok=True)

# ECMWF / Herbie settings
MODEL = "ifs"
PRODUCT = "oper"
tres=3
horz=72

# tres out to horz hours
FXX = list(range(0, horz, tres))

# Search string for 2-m temperature, 2-m dewpoint, and surface pressure
# Herbie uses the ECMWF GRIB index, so this selects only these GRIB messages.
SEARCH = r":(?:2t|2d|sp):"

# File names
COMBINED_NETCDF = OUTDIR / f"ecmwf_ifs_latest_surface_f000_f{horz:02d}.nc"

# If True, the notebook will not repeat downloads that already exist locally.
SKIP_EXISTING = True

## Obtain the latest available ECMWF forecast

Below, we use Herbie to find the recent ECMWF forecast cycle and download it. The code is only relevant if you think that you will want to download forecast data through Herbie in the future; otherwise it can be skipped (hence why it is hidden).

The output that prints to screen is important, though: it tells us the names of the variables in the dataset. We will need this to compute the humid heat quantities.

**Run the code below to grab the forcast data**.


In [ ]:
# @title
def find_latest_ecmwf_cycle(max_age_days=4):
    """
    Find the latest available ECMWF IFS forecast cycle.
    """
    now = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)

    for day_back in range(max_age_days + 1):
        day = now.date() - timedelta(days=day_back)

        for hour in [18, 12, 6, 0]:
            cycle = datetime(day.year, day.month, day.day, hour, tzinfo=timezone.utc)

            if cycle > now:
                continue

            try:
                H = Herbie(
                    cycle.strftime("%Y-%m-%d %H:%M"),
                    model=MODEL,
                    product=PRODUCT,
                    fxx=0,
                )

                if H.grib is not None:
                    print(f"Using ECMWF forecast cycle: {cycle:%Y-%m-%d %HZ}")
                    return cycle

            except Exception:
                continue

    raise RuntimeError("Could not find an available ECMWF forecast cycle.")


cycle = find_latest_ecmwf_cycle()

def retrieve_ecmwf_surface_forecast(cycle, fxx_list=FXX):
    """
    Download selected ECMWF surface fields and combine them into one xarray Dataset.
    """
    datasets = []

    for fxx in tqdm(fxx_list, desc="Downloading/opening ECMWF forecast"):
        try:
            H = Herbie(
                cycle.strftime("%Y-%m-%d %H:%M"),
                model=MODEL,
                product=PRODUCT,
                fxx=fxx,
                save_dir=OUTDIR,
            )

            path = Path(H.download(search=SEARCH))

            ds_i = xr.open_dataset(
                path,
                engine="cfgrib",
                backend_kwargs={"indexpath": ""},
            )

            ds_i = ds_i.expand_dims(
                valid_time=[pd.to_datetime(ds_i.valid_time.values)]
            )

            datasets.append(ds_i)

        except Exception as e:
            print(f"Skipping f{fxx:03d}: {e}")

    if not datasets:
        raise RuntimeError("No forecast files could be downloaded/opened.")

    ds = xr.concat(datasets, dim="valid_time").sortby("valid_time")

    _, unique_index = np.unique(ds.valid_time.values, return_index=True)
    ds = ds.isel(valid_time=sorted(unique_index))

    return ds

ds = retrieve_ecmwf_surface_forecast(cycle)
print("\n\nDOWNLOADED DATA!! \n\nSee data structure below:\n\n")
print(ds)
ds.to_netcdf(COMBINED_NETCDF) # Save to your drive (in case you want to reload
# later)

## Computing moist heat metrics
In this section we compute the moist heat metrics (almost) 'by hand' to illustrate the key principles. There are excellent Python modules available to compute relevant metrics (e.g., [Thermofeel](https://thermofeel.readthedocs.io/en/latest/guide/overview.html) and [MetPy](https://unidata.github.io/MetPy/latest/api/generated/metpy.calc.wet_bulb_temperature.html)), but we sometimes have little control over important methodological subtleties going down that route. For example, MetPy computes the pseudo-adiabatic, rather than the thermodynamic wet-bulb temperature; and Thermofeel uses an empirical approximation. Neither module computes equivalent temperature.

We therefore perform the following in the code below:


*  (a) **Compute specific and relative humidity** ($RH$) using dewpoint ($T_d$ and air temperature ($T_d$):

$$q =\frac{0.622\,e_s(T_d)}{p-0.378\,e_s(T_d)}\tag{1}$$

> in which the saturation vapour pressure ($e_s$) is computed with respect to a liquid water surface following [Huang (2018)](https://journals.ametsoc.org/view/journals/apme/57/6/jamc-d-17-0334.1.xml):

$$e_s(T)=611.21\,\exp\!\left[\left(18.678-\frac{T}{234.5}\right)\left(\frac{T}{257.14+T}\right)\right]\tag{2}$$

> where $T$ is the air temperature in $^\circ\mathrm{C}$ and $e_s$ is the saturation vapour pressure (Pa).

> $RH$ is computed simply from:

$$\frac{e_s(T_d)}{e_s(T_a)}\tag{3}$$

*  (b) **Calculate the specific heat capacity of moist air** ($C_pw$, which varies with humidity)--which is mass-weighted average of the dry-air and water-vapour heat capacities:

$$c_{pm} = (1-q) c_{pd} + q c_{pv}\tag{4}$$


* **(c) Evaluate the latent heat of vaporization**, which varies with $T_a$
following ([Henderson-Sellers, 1984](https://rmets.onlinelibrary.wiley.com/doi/10.1002/qj.49711046626)):

$$L_v(T) = 1.91846 \times 10^{6}\left(\frac{T}{T-33.91}\right)^2\tag{5}$$

* **(d) Compute the equivalent temperature ($T_e$):**

$$T_e=T_a+\frac{L_v q}{c_{pm}}\tag{6}$$

> The second term on the right-hand side is the 'latent tempeature' $(T_q$):

$$T_q=\frac{L_v q}{c_{pm}}\tag{7}$$

> Recall that it represents the temperature increase that would result if all atmospheric water vapour condensed and its latent heat were used to warm the air parcel. Equivalent temperature is simply the sum of the sensible ($T_a$) and latent ($T_q$) components of atmospheric heat.

* **(e) Compute the thermodynamic wet-bulb temperature ($T_w$)** by numerically solving the implicit moist enthalpy balance introduced earlier in the notebook. We do this using the approach described in [Romps (2026)](https://journals.ametsoc.org/view/journals/apme/65/2/JAMC-D-25-0130.1.xml) and made available in the  [heatindex Python module](https://pypi.org/project/heatindex/).

To save the amount of code in this workbook, we will import functions that evaluate these formulae from a separate file. But, **if you feel comfortable, please try calculating these yourself**. (Recall that you can add a Code cell (+ Code) if you want to try this.)

In [ ]:
# @title
!wget -q -O workshop_functions.py \
https://raw.githubusercontent.com/climatom/BioMetConference/main/src/workshop_functions.py

import workshop_functions as wf

# Compute RH from dewpoint and air temperature (note forecast in K, function
# requires C). Also note that for all array functions, we ravel() to create a
# vector, then we reshape to the orginal shape.
ds_shape = ds.t2m.shape

# ** (a) **
# Vapour pressure from dewpoint
e = wf.satvp_huang(ds.d2m.values.ravel() - 273.15).reshape(ds_shape)

# Saturation vapour pressure at air temperature
es = wf.satvp_huang(ds.t2m.values.ravel() - 273.15).reshape(ds_shape)

# Relative humidity [0-1]
rh = e / es

# Specific humidity [kg kg-1]
q = ((0.622 * e) / (ds.sp.values - 0.378 * e)).reshape(ds_shape)

# **(b) and (c) **
# Latent heat of vaporization [J kg-1] and specific heat capacity
# of moist air [J kg-1 K-1]
lv, cp= wf._moist_properties(ds.t2m.values.ravel(),q.ravel())
lv=lv.reshape(ds_shape)
cp=cp.reshape(ds_shape)

# ** (d) **
# Equivalent temperature [K]
te=ds.t2m+lv/cp*q

# ** (e) **
# Wet-bulb temperature [K]
# Note: call signature is hi.wetbulb(p,T,rh). And no ravel() needed.
tw=hi.wetbulb(ds.sp.values,ds.t2m.values,rh)

# Wrap computed quantities into the xarray Dataset
ds["e"] = xr.DataArray(
    e,
    coords=ds.t2m.coords,
    dims=ds.t2m.dims,
    attrs={"long_name": "vapour pressure", "units": "Pa"},
)

ds["es"] = xr.DataArray(
    es,
    coords=ds.t2m.coords,
    dims=ds.t2m.dims,
    attrs={"long_name": "saturation vapour pressure", "units": "Pa"},
)

ds["rh"] = xr.DataArray(
    rh,
    coords=ds.t2m.coords,
    dims=ds.t2m.dims,
    attrs={"long_name": "relative humidity", "units": "1"},
)

ds["q"] = xr.DataArray(
    q,
    coords=ds.t2m.coords,
    dims=ds.t2m.dims,
    attrs={"long_name": "specific humidity", "units": "kg kg-1"},
)

ds["lv"] = xr.DataArray(
    lv,
    coords=ds.t2m.coords,
    dims=ds.t2m.dims,
    attrs={"long_name": "latent heat of vaporization", "units": "J kg-1"},
)

ds["cp"] = xr.DataArray(
    cp,
    coords=ds.t2m.coords,
    dims=ds.t2m.dims,
    attrs={"long_name": "specific heat capacity of moist air", "units": "J kg-1 K-1"},
)

ds["te"] = xr.DataArray(
    te,
    coords=ds.t2m.coords,
    dims=ds.t2m.dims,
    attrs={"long_name": "equivalent temperature", "units": "K"},
)

ds["tw"] = xr.DataArray(
    tw,
    coords=ds.t2m.coords,
    dims=ds.t2m.dims,
    attrs={"long_name": "wet-bulb temperature", "units": "K"},
)

## Plotting the computed humid heat metrics

After running the code above, all of our computed quantities should have been added to the forecast xarray Dataset.

We will check this below by, first, printing the Dataset so that we can see the variable names to confirm what's there.

Second, we will make a quick plot of one field (by default, the *latent temperature, $T_q$*, field).

Please run the code below **at least twice**:

1. Without modification
2. Plotting a *different* variable calculated in the code above (and don't forget to change the colour bar label, plotting limits (vmin and vmax), and anything else that might need updating!)
3. Repeat 2 for as many of the new metrics that you want to explore!



In [ ]:
# @title
## Options
variable_to_plot="tw"
vmin=-50.
vmax=50.
offset=-273.15 # (To convert K to C, if needed; set to 0 otherwise)

latest_to_plot=ds[variable_to_plot].isel(valid_time=-1) + offset

# Quick plot of the latest wet-bulb temperature forecast
latest_to_plot.plot(
    x="longitude",
    y="latitude",
    cmap="seismic",
    vmin=vmin,
    vmax=vmax,
    cbar_kwargs={
        "label": "Wet-bulb temperature, $T_w$ [$^\circ$C]",
        "extend": "neither",
    },
)
plt.title(
    f"Latest forecast wet-bulb temperature\n"
    f"{pd.to_datetime(latest_to_plot.valid_time.values):%Y-%m-%d %H:%M UTC}"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

## Forecasting the *survival margin*

We are now ready to use the forecast to predict proximity to *unsurvivable* thresholds, which we'll call the *survival margin* ($S_m$).

First, though, we illustrate the principle. Running the code below will plot forecast grid cells from the latest timestep in temperature--relative-humidity state space, together with two definitions of unsurvivable heat thresholds ([Vanos et al., 2023](https://www.nature.com/articles/s41467-023-43121-5) and [Kearney et al., 2026](https://onlinelibrary.wiley.com/doi/10.1111/gcb.70830)). It also overlays contours in $T_e$.


In [ ]:
# @title
!wget -q -O survival_homo.txt \
https://raw.githubusercontent.com/climatom/BioMetConference/main/anc/survival_homo.txt

!wget -q -O survival_vanos.txt \
https://raw.githubusercontent.com/climatom/BioMetConference/main/anc/survival_vanos.txt

survival_homo = np.loadtxt("survival_homo.txt")
survival_vanos = np.loadtxt("survival_vanos.txt")

fig,ax=plt.subplots(1,1)
ax.plot(survival_homo[:,0],survival_homo[:,1],color='black',
        linewidth=3,label="Kearney [2026]")
ax.plot(survival_vanos[:,0],survival_vanos[:,1],color='black',
        linewidth=3,linestyle='--',label="Vanos [2023]")
xplot=ds["t2m"].isel(valid_time=-1).values-273.15
yplot=ds["rh"].isel(valid_time=-1).values*100.
ax.scatter(xplot.ravel(),yplot.ravel(),s=0.5,color='purple',alpha=0.05)
ax.set_xlim(10,60)
ax.set_ylim(5,100)
ax.grid()
ax.set_xlabel("Temperature, $T_a$ [$^\circ$C]")
ax.set_ylabel("Relative humidity, $RH$ [%]")

## Grid of Te
#Air temperature and relative humidity axes
ta_vals = np.linspace(5, 70, 500)      # degC
rh_vals = np.linspace(0, 100, 500)     # %

Ta_grid, RH_grid = np.meshgrid(ta_vals, rh_vals)

# Assume sea-level pressure
p0 = 1e5  # Pa

# Convert RH to vapour pressure and specific humidity
e_grid = wf.satvp_huang(Ta_grid.ravel()) * RH_grid.ravel() / 100.0
q_grid = (0.622 * e_grid) / (p0 - 0.378 * e_grid)

# Compute equivalent temperature
_, Te_grid = wf.me_te(Ta_grid.ravel() + 273.15, q_grid)

# Back to 2D, and convert to degC
Te_grid = Te_grid.reshape(Ta_grid.shape)
Te_grid_C = Te_grid - 273.15

# Plot on axes
cs = ax.contour(Ta_grid,RH_grid,Te_grid_C, levels=20,colors="0.35",linewidths=0.8,
    alpha=0.8,
)

ax.clabel(cs,fmt=lambda x: rf"$T_e={x:.0f}^\circ$C",fontsize=8,inline=True,
)
ax.legend()
plt.show()


Notice, in the above, that no forecast points cross even the lower (Vanos) threshold. In reality, the forecast survival margins are likely to be somewhat smaller because the plot is based on instantaneous atmospheric conditions, whereas the thresholds correspond to 6-hour average exposures.

A forecast survival margin ($S_m$) can be defined for every forecast time and every location (i.e., every scatter point) as the distance between the forecast atmospheric state and the relevant unsurvivable threshold.

As the figure illustrates, however, the definition of this *distance* is not unique: it depends on the direction in which it is measured through state space. For simplicity, we will consider only **horizontal** movements. In other words, if relative humidity remains fixed and only air temperature changes—a reasonable first-order approximation during many extreme heat events—how much additional heating is required to reach the threshold?

Even then, the choice of distance metric is not obvious. One possibility would be simply to measure the difference in air temperature ($^\circ$C). However, not all 1$^\circ$C increases represent the same increase in atmospheric heat content. At fixed relative humidity, progressively more energy is required to raise air temperature as the atmosphere becomes warmer. For this reason, my preferred distance metric is the equivalent temperature ($T_e$), which is proportional to moist enthalpy. The $T_e$-based survival margin therefore measures how much additional atmospheric heat content (expressed as an equivalent temperature) is required, assuming fixed relative humidity, before the unsurvivable threshold is reached.

If all of this sounds rather complicated, don't worry too much about the details. The key idea is that a $T_e$-based survival margin provides a physically motivated measure of how 'close' the atmosphere is to an empirical unsurvivable heat threshold.

The code below computes this distance for us using the Vanos threshold. First, it takes a running 6h average of the instantaneous forecasts, and it then computes $S_m$ using $T_e$. It then displays this as an animation. No product like this yet exists, but we might imagine that something like this could be needed as the climate continues to warm.

** Run the code now to compute and animate $S_m$.**



In [ ]:
# @title
# This code computes Sm as the Te distance to the thresholds

# Threshold line in Te space
p0 = 1e5
e_line = wf.satvp_huang(survival_vanos[:, 0]) * survival_vanos[:, 1] / 100.
q_line = (0.622 * e_line) / (p0 - 0.378 * e_line)

_, te_line = wf.me_te(survival_vanos[:, 0] + 273.15, q_line)

rh_ref = survival_vanos[:, 1].astype(np.float64)
te_ref = te_line.astype(np.float64)

# Sort reference line by RH
idx = np.argsort(rh_ref)
rh_ref = rh_ref[idx]
te_ref = te_ref[idx]

# 6-h rolling mean from 3-hourly data
ds_6_valid = (
    ds
    .rolling(valid_time=3, min_periods=3)
    .mean()
    .isel(valid_time=slice(2, None))
    .transpose("valid_time", "latitude", "longitude")
)

# Compute Sm / dTe
dte = wf.delta_te(
    (ds_6_valid.rh.values * 100.).astype(np.float64),
    ds_6_valid.te.values.astype(np.float64),
    rh_ref,
    te_ref,
)

# Animation setup
lon = ds_6_valid.longitude.values
lat = ds_6_valid.latitude.values
times = ds_6_valid.valid_time.values

vmin = -5
vmax = 25

dte_plot = np.where(dte > vmax, np.nan, dte)

fig = plt.figure(figsize=(8, 7))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.coastlines(resolution="10m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

gl = ax.gridlines(
    draw_labels=True,
    linewidth=0.3,
    color="0.5",
    alpha=0.5,
    linestyle="--",
)
gl.top_labels = False
gl.right_labels = False

mesh = ax.pcolormesh(
    lon,
    lat,
    np.ma.masked_invalid(dte_plot[0]),
    cmap="magma",
    shading="nearest",      # easier to update than "auto"
    vmin=vmin,
    vmax=vmax,
    transform=ccrs.PlateCarree(),
)

cb = plt.colorbar(mesh, ax=ax, pad=0.02, shrink=0.85, extend="both")
cb.set_label(r"$S_m$: $T_e$ margin to Vanos 6-h threshold [°C]")

title = ax.set_title("", fontsize=13)

def update(frame):
    mesh.set_array(np.ma.masked_invalid(dte_plot[frame]).ravel())

    t = np.datetime_as_string(times[frame], unit="h")
    title.set_text(f"6-hour mean $S_m$\n{t} UTC")

    return mesh, title

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(times),
    interval=350,
    blit=False,
)

plt.close(fig)
HTML(ani.to_jshtml())

## Reflection

Inspect both the animation and the earlier scatter plot (with $T_e$) contours. Which regions have the smallest $S_m$, and what does this tell us about the importance, or not, of assessing *humid heat* metrics, rather than air temperature alone?

> (As a question to prompt this reflection, discuss in pairs whether 'dry' or 'humid' regions have smaller $S_m$).